# Arrow tables reach Spark without dropping rows

**Purpose:** `spark.createDataFrame(pyarrow.Table)` has stopped at the first zero-row record batch on
PySpark 4.0 (classic, local), silently dropping every row after it. This probe builds a table with
empty chunks (as `pa.concat_tables` leaves when a part is empty) and reports, on this runtime:

- how many rows the raw `createDataFrame` keeps (informational: it does not fail the probe);
- that `databricks.labs.gbx.core._arrow.to_spark_df` keeps all of them (asserted).

**Serverless / JOB usage:** no `%pip` cells; the runner injects the wheel into the environment spec
(`gbx:test:notebooks-serverless --notebook examples/tests/probe_arrow_empty_chunks.ipynb --extras vizx`).
The result is returned through `dbutils.notebook.exit`.

In [ ]:
import json

import pyarrow as pa
import pyspark
from pyspark.sql import types as T

from databricks.labs.gbx.core._arrow import to_spark_df


def ints(*values):
    return pa.table({"k": pa.array(values, pa.int64())})


empty = pa.table({"k": pa.array([], pa.int64())})
table = pa.concat_tables([empty, ints(1, 2), empty, ints(3, 4, 5), empty, ints(6)])
WANT = [1, 2, 3, 4, 5, 6]

results = {
    "pyspark": pyspark.__version__,
    "session_class": f"{type(spark).__module__}.{type(spark).__name__}",
    "chunk_rows": [len(c) for c in table.column("k").chunks],
    "want": WANT,
}
print(json.dumps(results, indent=2))
assert results["chunk_rows"].count(0) == 3, "the probe table must carry zero-row chunks"

In [ ]:
# The raw call: informational. If this runtime loses rows, the helper is needed here.
try:
    raw = sorted(r["k"] for r in spark.createDataFrame(table).collect())
    results["raw_rows"] = raw
    results["raw_drops_rows"] = raw != WANT
except Exception as exc:  # report, do not fail: the question is what the runtime does
    results["raw_error"] = f"{type(exc).__name__}: {str(exc)[:300]}"
    results["raw_drops_rows"] = None
print("raw createDataFrame:", results.get("raw_rows", results.get("raw_error")))

In [ ]:
# The helper: asserted. Inferred schema, then an explicit StructType.
schema = T.StructType([T.StructField("k", T.LongType(), True)])

inferred = sorted(r["k"] for r in to_spark_df(spark, table).collect())
explicit_df = to_spark_df(spark, table, schema)
explicit = sorted(r["k"] for r in explicit_df.collect())
all_empty = to_spark_df(spark, pa.concat_tables([empty, empty]), schema).count()

results["helper_inferred"] = inferred
results["helper_explicit"] = explicit
results["helper_all_empty_count"] = all_empty
results["pass"] = inferred == WANT and explicit == WANT and all_empty == 0
print(json.dumps(results, indent=2))
assert inferred == WANT, f"to_spark_df (inferred schema) lost rows: {inferred}"
assert explicit == WANT, f"to_spark_df (explicit schema) lost rows: {explicit}"
assert all_empty == 0, f"an all-empty table gave {all_empty} rows"
print("\nPASS: to_spark_df kept all", len(WANT), "rows; raw createDataFrame drops rows:", results["raw_drops_rows"])
dbutils.notebook.exit(json.dumps(results))  # surface the result to jobs.get_run_output